# ⚽ Fußball Match Outcome Predictor – Datenpipeline

**Projekt:** KI & Intelligence Engineering  
**Institution:** DHBW Mannheim  

## 📌 Beschreibung

Dieses Skript bildet die Datenpipeline für einen Fußball-Match-Outcome-Predictor.  
Es bereitet historische Spieldaten für das anschließende Modelltraining auf.

Die Pipeline umfasst folgende Schritte:

1. **Datenimport**
   - Lädt historische internationale Fußballergebnisse direkt von GitHub.
   - Datenquelle: [`martj42/international_results`](https://github.com/martj42/international_results)

2. **Elo-Rating-Berechnung**
   - Berechnet dynamische Teamstärken mithilfe des Elo-Systems.
   - Berücksichtigt die historische Performance der Mannschaften.

3. **Feature Engineering**
   - Erstellt relevante Features für Machine-Learning-Modelle.
   - Bereitet die Daten für das Training des Predictors vor.

4. **Datenspeicherung**
   - Speichert den aufbereiteten Datensatz für die weitere Verarbeitung.

---

## 🚀 Verwendung

Das Skript kann direkt über die Kommandozeile ausgeführt werden:

```bash
python datenpipeline.py


**imports**

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

**Konfig**

In [10]:
START_YEAR = 1994
ROLLING_WINDOW = 5  # Formkurve über letzte 5 Spiele
OUTPUT_DIR = "data/processed_v2"

# GitHub-URL für den Datensatz (martj42 – wird regelmäßig aktualisiert)
DATA_URL = "https://raw.githubusercontent.com/martj42/international_results/master/results.csv"

print("=" * 60)
print("⚽  Fußball Match Outcome Predictor – Datenpipeline v2")
print("=" * 60)
print(f"Startjahr: {START_YEAR} | Rolling Window: {ROLLING_WINDOW} Spiele")
print(f"Datenquelle: {DATA_URL}")

⚽  Fußball Match Outcome Predictor – Datenpipeline v2
Startjahr: 1994 | Rolling Window: 5 Spiele
Datenquelle: https://raw.githubusercontent.com/martj42/international_results/master/results.csv


# Lade Datensatz von Github und filtere

In [11]:
print("📂 Lade Datensatz...")
matches = pd.read_csv(DATA_URL, parse_dates=["date"])
matches['year'] = matches['date'].dt.year
matches = matches[matches['year'] >= START_YEAR].copy()
matches = matches.sort_values('date').reset_index(drop=True)

print(f"✅ {len(matches):,} Spiele ab {START_YEAR}")

# Zielvariable
matches['target'] = np.where(
    matches['home_score'] > matches['away_score'], 0,
    np.where(matches['home_score'] == matches['away_score'], 1, 2)
)

📂 Lade Datensatz...
✅ 30,017 Spiele ab 1994


# EDA & Datapreprocessing

In [12]:
print("" + "─" * 60)
print("📊 Schritt 2: Daten-Exploration")
print("─" * 60)

matches['year'] = matches['date'].dt.year

print(f"Zeitraum: {matches['year'].min()} – {matches['year'].max()}")
print(f"Einzigartige Teams: {matches['home_team'].nunique()}")
print(f"Turniere: {matches['tournament'].nunique()}")

print(f"Top 10 Turniere:")
print(matches['tournament'].value_counts().head(10))

# Zielvariable erstellen
matches['target'] = matches.apply(
    lambda row: 0 if row['home_score'] > row['away_score']
    else (1 if row['home_score'] == row['away_score'] else 2),
    axis=1
)

print(f"📈 Ergebnisverteilung:")
print(matches['target'].value_counts(normalize=True).round(3))
print("   0 = Heimsieg | 1 = Unentschieden | 2 = Auswärtssieg")

────────────────────────────────────────────────────────────
📊 Schritt 2: Daten-Exploration
────────────────────────────────────────────────────────────
Zeitraum: 1994 – 2026
Einzigartige Teams: 316
Turniere: 141
Top 10 Turniere:
tournament
Friendly                                10018
FIFA World Cup qualification             6482
UEFA Euro qualification                  1991
African Cup of Nations qualification     1691
UEFA Nations League                       658
AFC Asian Cup qualification               613
African Cup of Nations                    606
FIFA World Cup                            604
CFU Caribbean Cup qualification           433
CONCACAF Nations League                   422
Name: count, dtype: int64
📈 Ergebnisverteilung:
target
0    0.485
2    0.282
1    0.233
Name: proportion, dtype: float64
   0 = Heimsieg | 1 = Unentschieden | 2 = Auswärtssieg


# Elo-Ratings berechnen

In [13]:
print("⚡ Berechne Elo-Ratings...")

ratings = {}
home_elos, away_elos = [], []

for _, row in matches.iterrows():
    home, away = row['home_team'], row['away_team']

    if home not in ratings: ratings[home] = 1500
    if away not in ratings: ratings[away] = 1500

    h_r, a_r = ratings[home], ratings[away]
    home_elos.append(h_r)
    away_elos.append(a_r)

    E_h = 1 / (1 + 10 ** ((a_r - h_r) / 400))

    if row['home_score'] > row['away_score']:
        S_h = 1
    elif row['home_score'] == row['away_score']:
        S_h = 0.5
    else:
        S_h = 0

    ratings[home] += 40 * (S_h - E_h)
    ratings[away] += 40 * ((1 - S_h) - (1 - E_h))

matches['home_elo'] = home_elos
matches['away_elo'] = away_elos
matches['elo_diff'] = matches['home_elo'] - matches['away_elo']
matches['elo_diff_sq'] = matches['elo_diff'] ** 2
matches['elo_diff_abs'] = matches['elo_diff'].abs()
matches['is_neutral'] = matches['neutral'].astype(int)
matches['elo_diff_x_neutral'] = matches['elo_diff'] * matches['is_neutral']

print("✅ Elo-Ratings + nicht-lineare Features")

⚡ Berechne Elo-Ratings...
✅ Elo-Ratings + nicht-lineare Features


# Turnier-Importanz

In [14]:
def tournament_importance(t):
    t = t.lower()
    if 'world cup' in t: return 5
    elif 'euro' in t: return 4
    elif 'confederations' in t: return 4
    elif 'nations league' in t: return 3
    elif 'qualification' in t or 'qualifier' in t: return 2
    elif 'friendly' in t: return 1
    else: return 2

matches['tournament_importance'] = matches['tournament'].apply(tournament_importance)
print("✅ Turnier-Importanz")

✅ Turnier-Importanz


# Team Statistiken in einer Liste für bessere Effizienz

In [15]:
print("🔧 Berechne Team-Statistiken (effizient)...")

# Erstelle lange Liste: jedes Spiel = 2 Einträge (Home + Away)
records = []
for idx, row in matches.iterrows():
    date = row['date']

    if row['home_score'] > row['away_score']:
        h_pts, a_pts = 3, 0
        h_res, a_res = 'W', 'L'
    elif row['home_score'] == row['away_score']:
        h_pts, a_pts = 1, 1
        h_res, a_res = 'D', 'D'
    else:
        h_pts, a_pts = 0, 3
        h_res, a_res = 'L', 'W'

    # Speichere Original-Index für spätere Zuordnung
    records.append({
        'match_idx': idx, 'team': row['home_team'], 'is_home': True,
        'date': date, 'pts': h_pts, 'res': h_res,
        'scored': row['home_score'], 'conceded': row['away_score'],
        'elo': row['home_elo']
    })
    records.append({
        'match_idx': idx, 'team': row['away_team'], 'is_home': False,
        'date': date, 'pts': a_pts, 'res': a_res,
        'scored': row['away_score'], 'conceded': row['home_score'],
        'elo': row['away_elo']
    })

team_df = pd.DataFrame(records).sort_values(['team', 'date']).reset_index(drop=True)

# --- Rolling Stats pro Team ---
for w in [3, 5, 10]:
    team_df[f'form_{w}'] = team_df.groupby('team')['pts'].transform(
        lambda x: x.rolling(w, min_periods=1).mean().shift(1)
    )

for w in [5, 10]:
    team_df[f'avg_scored_{w}'] = team_df.groupby('team')['scored'].transform(
        lambda x: x.rolling(w, min_periods=1).mean().shift(1)
    )
    team_df[f'avg_conceded_{w}'] = team_df.groupby('team')['conceded'].transform(
        lambda x: x.rolling(w, min_periods=1).mean().shift(1)
    )

# Tage seit letztem Spiel
team_df['days_rest'] = team_df.groupby('team')['date'].diff().dt.days

# Erfahrung (Spielzahl)
team_df['experience'] = team_df.groupby('team').cumcount()

# Elo-Momentum (Trend letzte 5)
team_df['elo_change'] = team_df.groupby('team')['elo'].diff()
team_df['elo_momentum'] = team_df.groupby('team')['elo_change'].transform(
    lambda x: x.rolling(5, min_periods=1).mean().shift(1)
)

# Siegesserie
def streak_length(group):
    streak = []
    curr = 0
    last = None
    for r in group:
        if r == last:
            curr += 1
        else:
            curr = 1
            last = r
        streak.append(curr)
    return streak

team_df['streak'] = team_df.groupby('team')['res'].transform(streak_length)
team_df['streak'] = team_df.groupby('team')['streak'].shift(1)

# Streak-Typ kodieren
type_map = {'W': 2, 'D': 1, 'L': 0}
team_df['streak_type'] = team_df.groupby('team')['res'].transform(
    lambda x: x.map(type_map).shift(1)
)

print("✅ Team-Statistiken berechnet")

🔧 Berechne Team-Statistiken (effizient)...
✅ Team-Statistiken berechnet


# Zurück zu Matches (Match-Zuordnung)

In [16]:
print("📋 Ordne Features zu...")

# Splitte in Home/Away
team_df = team_df.sort_values('match_idx')
home_df = team_df[team_df['is_home'] == True].set_index('match_idx').sort_index()
away_df = team_df[team_df['is_home'] == False].set_index('match_idx').sort_index()

# Home-Features
matches['home_form_3'] = home_df['form_3'].values
matches['home_form_5'] = home_df['form_5'].values
matches['home_form_10'] = home_df['form_10'].values
matches['home_avg_scored_5'] = home_df['avg_scored_5'].values
matches['home_avg_conceded_5'] = home_df['avg_conceded_5'].values
matches['home_avg_scored_10'] = home_df['avg_scored_10'].values
matches['home_avg_conceded_10'] = home_df['avg_conceded_10'].values
matches['home_days_rest'] = home_df['days_rest'].values
matches['home_experience'] = home_df['experience'].values
matches['home_elo_momentum'] = home_df['elo_momentum'].values
matches['home_streak'] = home_df['streak'].values
matches['home_streak_type'] = home_df['streak_type'].values

# Away-Features
matches['away_form_3'] = away_df['form_3'].values
matches['away_form_5'] = away_df['form_5'].values
matches['away_form_10'] = away_df['form_10'].values
matches['away_avg_scored_5'] = away_df['avg_scored_5'].values
matches['away_avg_conceded_5'] = away_df['avg_conceded_5'].values
matches['away_avg_scored_10'] = away_df['avg_scored_10'].values
matches['away_avg_conceded_10'] = away_df['avg_conceded_10'].values
matches['away_days_rest'] = away_df['days_rest'].values
matches['away_experience'] = away_df['experience'].values
matches['away_elo_momentum'] = away_df['elo_momentum'].values
matches['away_streak'] = away_df['streak'].values
matches['away_streak_type'] = away_df['streak_type'].values

# Differenz-Features
matches['goal_balance_home'] = matches['home_avg_scored_5'] - matches['home_avg_conceded_5']
matches['goal_balance_away'] = matches['away_avg_scored_5'] - matches['away_avg_conceded_5']
matches['goal_balance_diff'] = matches['goal_balance_home'] - matches['goal_balance_away']
matches['rest_diff'] = matches['home_days_rest'] - matches['away_days_rest']
matches['experience_diff'] = matches['home_experience'] - matches['away_experience']
matches['elo_momentum_diff'] = matches['home_elo_momentum'] - matches['away_elo_momentum']

# Fehlende Werte füllen
fill_cols = [
    'home_form_3', 'home_form_5', 'home_form_10',
    'away_form_3', 'away_form_5', 'away_form_10',
    'home_avg_scored_5', 'home_avg_conceded_5', 'home_avg_scored_10', 'home_avg_conceded_10',
    'away_avg_scored_5', 'away_avg_conceded_5', 'away_avg_scored_10', 'away_avg_conceded_10',
    'home_days_rest', 'away_days_rest',
    'home_elo_momentum', 'away_elo_momentum',
    'home_streak', 'away_streak', 'home_streak_type', 'away_streak_type'
]
for col in fill_cols:
    matches[col] = matches[col].fillna(1.0 if 'form' in col or 'avg' in col else (0 if 'streak' in col else 30))

print("✅ Alle Features zugeordnet")

📋 Ordne Features zu...
✅ Alle Features zugeordnet


# H2H

In [17]:
print("🤝 Berechne Head-to-Head...")

matches['pair_id'] = matches.apply(
    lambda r: '|||'.join(sorted([r['home_team'], r['away_team']])), axis=1
)

# Gruppiere und berechne H2H
h2h_wins = []
h2h_draws = []
h2h_losses = []

for _, group in matches.groupby('pair_id'):
    group = group.sort_values('date')
    wins = draws = losses = 0

    for _, row in group.iterrows():
        h2h_wins.append(wins)
        h2h_draws.append(draws)
        h2h_losses.append(losses)

        if row['home_score'] > row['away_score']:
            wins += 1
        elif row['home_score'] == row['away_score']:
            draws += 1
        else:
            losses += 1

# Zuordnung per Index (groupby behält Reihenfolge bei)
h2h_idx = []
for _, group in matches.groupby('pair_id'):
    h2h_idx.extend(group.index.tolist())

matches.loc[h2h_idx, 'h2h_home_wins'] = h2h_wins
matches.loc[h2h_idx, 'h2h_draws'] = h2h_draws
matches.loc[h2h_idx, 'h2h_home_losses'] = h2h_losses

matches['h2h_total'] = matches['h2h_home_wins'] + matches['h2h_draws'] + matches['h2h_home_losses']
matches['h2h_win_rate'] = np.where(matches['h2h_total'] > 0, matches['h2h_home_wins'] / matches['h2h_total'], 0.33)

print("✅ Head-to-Head")

🤝 Berechne Head-to-Head...
✅ Head-to-Head


# Finale Feature-Matrix

In [18]:
print("📋 Finale Feature-Matrix...")

feature_columns = [
    'home_elo', 'away_elo', 'elo_diff', 'elo_diff_sq', 'elo_diff_abs', 'elo_diff_x_neutral',
    'home_elo_momentum', 'away_elo_momentum', 'elo_momentum_diff',
    'home_form_3', 'away_form_3', 'home_form_5', 'away_form_5', 'home_form_10', 'away_form_10',
    'home_avg_scored_5', 'home_avg_conceded_5', 'away_avg_scored_5', 'away_avg_conceded_5',
    'home_avg_scored_10', 'home_avg_conceded_10', 'away_avg_scored_10', 'away_avg_conceded_10',
    'goal_balance_home', 'goal_balance_away', 'goal_balance_diff',
    'h2h_home_wins', 'h2h_draws', 'h2h_home_losses', 'h2h_win_rate', 'h2h_total',
    'tournament_importance', 'is_neutral',
    'home_days_rest', 'away_days_rest', 'rest_diff',
    'home_experience', 'away_experience', 'experience_diff',
    'home_streak', 'away_streak', 'home_streak_type', 'away_streak_type',
]

target_col = 'target'
meta_cols = ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament']

final_df = matches[meta_cols + feature_columns + [target_col]].copy()
final_df = final_df.dropna(subset=feature_columns)

print(f"✅ Shape: {final_df.shape} | Features: {len(feature_columns)}")


📋 Finale Feature-Matrix...
✅ Shape: (29506, 50) | Features: 43


# Train/Test-Split nach Zeit

In [19]:
print("📅 Zeitlicher Split...")

TRAIN_END = '2018-01-01'
VAL_END = '2022-01-01'

train_df = final_df[final_df['date'] < TRAIN_END].copy()
val_df = final_df[(final_df['date'] >= TRAIN_END) & (final_df['date'] < VAL_END)].copy()
test_df = final_df[final_df['date'] >= VAL_END].copy()

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")

📅 Zeitlicher Split...
Train: 21,320 | Val: 3,523 | Test: 4,663


# Speichern

In [20]:
print("💾 Speichern...")

os.makedirs(OUTPUT_DIR, exist_ok=True)

final_df.to_csv(f"{OUTPUT_DIR}/matches_full.csv", index=False)
train_df.to_csv(f"{OUTPUT_DIR}/train.csv", index=False)
val_df.to_csv(f"{OUTPUT_DIR}/val.csv", index=False)
test_df.to_csv(f"{OUTPUT_DIR}/test.csv", index=False)
pd.DataFrame({'feature': feature_columns}).to_csv(f"{OUTPUT_DIR}/features.csv", index=False)

print(f"✅ Gespeichert in {OUTPUT_DIR}/")

print("" + "=" * 60)
print("✅  Datenpipeline v2 (opt) abgeschlossen!")
print("=" * 60)
print(f"Features: {len(feature_columns)}")
print("Nächster Schritt: python modelltraining_v2.py")

💾 Speichern...
✅ Gespeichert in data/processed_v2/
✅  Datenpipeline v2 (opt) abgeschlossen!
Features: 43
Nächster Schritt: python modelltraining_v2.py


# Zusammenfassung

In [21]:
print("" + "=" * 60)
print("✅  Datenpipeline v2 (opt) abgeschlossen!")
print("=" * 60)
print("""
Nächste Schritte:
  1. Modelltraining:  python modelltraining.py
  2. Streamlit-App:   streamlit run app.py
""")

✅  Datenpipeline v2 (opt) abgeschlossen!

Nächste Schritte:
  1. Modelltraining:  python modelltraining.py
  2. Streamlit-App:   streamlit run app.py

